# Supervised Learning with Quantum-Enhanced Feature Space

**Implementation of Havlicek *et al.*, Nature 567, 209–212 (2019)**

This notebook implements both classification strategies proposed in the paper:
1. **Quantum Kernel Estimator** — computes a quantum kernel matrix and feeds it to a classical SVM
2. **Quantum Variational Classifier** — trains a parametrized quantum circuit with classical optimization

Both methods encode classical data into quantum states via a **quantum feature map** $\mathcal{U}_{\Phi(\mathbf{x})}$, which maps data $\mathbf{x}$ to a quantum state $|\Phi(\mathbf{x})\rangle\langle\Phi(\mathbf{x})|$ in the Hilbert space. The key insight is that feature maps based on entangling gates at depth $d \geq 2$ are conjectured to be classically hard to simulate, potentially offering a quantum advantage.

---
## Part 0: Setup & Imports

### Helper Functions

---
## Part 1: Quantum Feature Maps — Theory & Exploration

The quantum feature map is the core of both classification strategies in the paper. It nonlinearly maps a classical datum $\mathbf{x}$ to a quantum state $|\Phi(\mathbf{x})\rangle\langle\Phi(\mathbf{x})|$ in the Hilbert space. The unitary implementing this map is:

$$\mathcal{U}_{\Phi(\mathbf{x})} = \prod_d U_{\Phi(\mathbf{x})} H^{\otimes n}, \qquad U_{\Phi(\mathbf{x})} = \exp\left(i \sum_{S \subseteq [n]} \phi_S(\mathbf{x}) \prod_{k \in S} P_k\right)$$

where:
- $d$ is the circuit **depth** (number of repetitions)
- $n$ is the number of qubits (equals the data dimensionality)
- $S \subseteq [n]$ indexes subsets of qubits; $|S| \leq r$ gives the **expansion order** $r$
- $\phi_S(\mathbf{x})$ are **data map functions** encoding the classical data
- $P_k \in \{\mathbb{1}, X, Y, Z\}$ are Pauli operators on qubit $k$

The **second-order expansion** ($r=2$) with $P_k = Z$ and depth $d=2$ is the feature map used in the paper. It is conjectured to be classically hard to simulate, while the first-order ($r=1$) variant can be efficiently simulated classically.

---
## Part 2: Quantum Kernel Estimator — Direct Kernel Method

The Quantum Kernel Estimator computes the kernel matrix $K_{ij} = |\langle\Phi(\mathbf{x}_i)|\Phi(\mathbf{x}_j)\rangle|^2$ using a quantum circuit, then feeds this matrix into a classical support vector machine (SVM). The quantum advantage comes from the feature map: if $\mathcal{U}_{\Phi(\mathbf{x})}$ is hard to simulate classically, then the kernel $K$ encodes correlations that are classically intractable to compute.

The algorithm proceeds as follows:
1. For each pair $(\mathbf{x}_i, \mathbf{x}_j)$ in the training set, compute $K_{ij}$ by running the "compute-uncompute" circuit: prepare $|\Phi(\mathbf{x}_i)\rangle$ and $|\Phi(\mathbf{x}_j)\rangle$ on separate registers, then measure the swap-like fidelity
2. Use $K$ as a precomputed kernel for a classical SVM (e.g., `QSVC` in Qiskit ML)
3. For prediction, compute kernel values between test points and training points, then apply the SVM decision function

We now apply this method to the **actual datasets from the paper** (Sets I, II, III).

### 2.1 Load Paper Datasets

The paper provides three dataset splits (Sets I, II, III) for the kernel method, each with 20 training samples (10 per class) and test samples. The data lives in the `data/` directory.

In [ ]:
# 2.1 Load the three dataset splits for the Direct Kernel method
kernel_sets = {}
for set_name in ["I", "II", "III"]:
    train_path = os.path.join(DATA_DIR, f"Direct_Kernel_Set_{set_name}_Training.csv")
    test_path = os.path.join(DATA_DIR, f"Direct_Kernel_Set_{set_name}_Classifications_ResultsOnly.csv")
    
    X_train, y_train = load_paper_csv(train_path)
    X_test, y_true, y_pred_paper = load_paper_classification(test_path)
    
    kernel_sets[set_name] = {
        "X_train": X_train, "y_train": y_train,
        "X_test": X_test, "y_test": y_true, "y_pred_paper": y_pred_paper
    }
    
    paper_acc = accuracy_score(y_true, y_pred_paper)
    print(f"Set {set_name}: train={X_train.shape[0]} samples, test={X_test.shape[0]} samples, "
          f"paper reported accuracy={paper_acc:.4f}")

# Visualize one of the training sets
plot_data(kernel_sets["I"]["X_train"], kernel_sets["I"]["y_train"],
          title="Paper Dataset — Direct Kernel Set I (Training)")

### 2.2 Quantum Kernel Estimation on Paper Data

We now compute the quantum kernel using `ZZFeatureMap` with depth $d=2$ (the paper's configuration) and train `QSVC` on each dataset split.

In [ ]:
# 2.2 Quantum Kernel Estimation on paper data using ZZFeatureMap (d=2)
zz_kernel_map = ZZFeatureMap(feature_dimension=2, reps=2, entanglement='full')

sampler = StatevectorSampler()
fidelity = ComputeUncompute(sampler=sampler)
qk = FidelityQuantumKernel(feature_map=zz_kernel_map, fidelity=fidelity)

kernel_results = {}
for set_name in ["I", "II", "III"]:
    data = kernel_sets[set_name]
    X_train, y_train = data["X_train"], data["y_train"]
    X_test, y_test = data["X_test"], data["y_test"]
    
    # Compute kernel matrices
    kernel_train = qk.evaluate(X_train)
    kernel_test = qk.evaluate(X_test, X_train)
    
    # Train QSVC
    qsvc = QSVC(quantum_kernel=qk)
    qsvc.fit(X_train, y_train)
    our_acc = qsvc.score(X_test, y_test)
    paper_acc = accuracy_score(y_test, data["y_pred_paper"])
    
    kernel_results[set_name] = {
        "our_accuracy": our_acc,
        "paper_accuracy": paper_acc,
        "qsvc": qsvc,
        "kernel_train": kernel_train
    }
    
    print(f"Set {set_name}: Our QSVC accuracy = {our_acc:.4f}, Paper accuracy = {paper_acc:.4f}")

# Visualize kernel matrix for Set I
plot_kernel_matrix(kernel_results["I"]["kernel_train"],
                   title="Quantum Kernel Matrix — Paper Set I (ZZFeatureMap d=2)")

# Decision boundary for Set I
plot_decision_boundary(kernel_results["I"]["qsvc"],
                       kernel_sets["I"]["X_train"], kernel_sets["I"]["y_train"],
                       title=f"Decision Boundary — Kernel Set I (acc={kernel_results['I']['our_accuracy']:.2f})")

### 2.3 Kernel Method — Results Comparison with Paper

Let's compare our quantum kernel results against the paper's reported classifications across all three dataset splits.

In [ ]:
# 2.3 Comparison table: our QSVC vs paper's reported results
print("=" * 65)
print(f"{'Set':<8} {'Our QSVC Acc':>15} {'Paper Acc':>15} {'Match?':>10}")
print("=" * 65)
for set_name in ["I", "II", "III"]:
    our = kernel_results[set_name]["our_accuracy"]
    paper = kernel_results[set_name]["paper_accuracy"]
    match = "✓" if abs(our - paper) < 0.05 else "✗"
    print(f"{set_name:<8} {our:>15.4f} {paper:>15.4f} {match:>10}")
print("=" * 65)

# Visualize all three decision boundaries side by side
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for idx, set_name in enumerate(["I", "II", "III"]):
    data = kernel_sets[set_name]
    qsvc = kernel_results[set_name]["qsvc"]
    X, y = data["X_train"], data["y_train"]
    
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 30), np.linspace(y_min, y_max, 30))
    Z = qsvc.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    axes[idx].contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.RdBu)
    axes[idx].scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.RdBu, edgecolors="k", s=40)
    acc = kernel_results[set_name]["our_accuracy"]
    axes[idx].set_title(f"Set {set_name} (acc={acc:.2f})", fontsize=12)
    axes[idx].set_xlabel("Feature $x_1$")
    axes[idx].set_ylabel("Feature $x_2$")
plt.suptitle("Quantum Kernel Estimator — Decision Boundaries (ZZFeatureMap d=2)", fontsize=14)
plt.tight_layout()
plt.show()

---
## Part 3: Quantum Variational Classifier

The second strategy from the paper is the **Quantum Variational Classifier**. Instead of computing a full kernel matrix, this approach trains a parametrized quantum circuit $W(\boldsymbol{\theta})$ that follows the feature map:

$$|\psi(\mathbf{x}; \boldsymbol{\theta})\rangle = W(\boldsymbol{\theta}) \mathcal{U}_{\Phi(\mathbf{x})} |0\rangle^{\otimes n}$$

The circuit is measured in the computational basis, and the measurement outcomes are mapped to class labels via an assignment function. The cost function is the **empirical risk**:

$$\mathcal{L}(\boldsymbol{\theta}) = \frac{1}{M} \sum_{i=1}^{M} \ell\big(h(\mathbf{x}_i; \boldsymbol{\theta}), y_i\big)$$

where $\ell$ is a loss function (e.g., cross-entropy) and $h(\mathbf{x}_i; \boldsymbol{\theta})$ is the classifier output. The parameters $\boldsymbol{\theta}$ are optimized classically (e.g., via COBYLA or SPSA).

The paper studies variational classifiers at depths $d = 0, 1, 2, 3, 4$ and finds that:
- $d = 0$ (feature map only, no variational ansatz) performs poorly
- $d = 1$ is classically simulable
- $d = 2$ achieves the best accuracy (the "sweet spot")
- $d \geq 3$ does not improve accuracy and is more noise-prone on hardware

### 3.1 Load Variational Datasets

The paper provides training and classification data for the variational method at depths $d = 0, 1, 2, 3, 4$ across three dataset splits (Sets I, II, III). Each training file includes a "Training Error" column.

In [ ]:
# 3.1 Load variational datasets for all depths and splits
depths = [0, 1, 2, 3, 4]
set_names = ["I", "II", "III"]

variational_data = {}
for set_name in set_names:
    for d in depths:
        train_path = os.path.join(DATA_DIR, f"Variational_Set_{set_name}_d{d}_Training.csv")
        test_path = os.path.join(DATA_DIR, f"Variational_Set_{set_name}_d{d}_Classifications_ResultsOnly.csv")
        
        # Training data
        df_train = pd.read_csv(train_path, header=0, index_col=0)
        df_train.columns = ["feature_1", "feature_2", "label"] + list(df_train.columns[3:])
        X_train = df_train[["feature_1", "feature_2"]].values
        y_train = df_train["label"].values.astype(int)
        
        # Extract training error if present
        train_error = None
        if "Training Error" in df_train.columns:
            train_error = df_train["Training Error"].dropna().values
            if len(train_error) > 0:
                train_error = train_error[0]
        
        # Test data with paper's predictions
        X_test, y_test, y_pred_paper = load_paper_classification(test_path)
        paper_acc = accuracy_score(y_test, y_pred_paper)
        
        variational_data[(set_name, d)] = {
            "X_train": X_train, "y_train": y_train,
            "X_test": X_test, "y_test": y_test,
            "y_pred_paper": y_pred_paper,
            "paper_accuracy": paper_acc,
            "training_error": train_error
        }

print("Loaded variational data for all depths and splits.")
print(f"Total configurations: {len(variational_data)} (3 sets × 5 depths)")

# Show a sample
sample = variational_data[("I", 2)]
print(f"\nSample — Set I, d=2:")
print(f"  Training samples: {sample['X_train'].shape[0]}")
print(f"  Test samples:     {sample['X_test'].shape[0]}")
print(f"  Paper accuracy:   {sample['paper_accuracy']:.4f}")
print(f"  Training error:   {sample['training_error']}")

### 3.2 VQC Training — Depth Sweep

We train the `VQC` (Variational Quantum Classifier) for each depth $d$ on each dataset split. The configuration follows the paper:

- **Feature map**: `ZZFeatureMap(feature_dimension=2, reps=d)` — the depth $d$ controls how many times the encoding circuit is repeated
- **Ansatz**: `RealAmplitudes(num_qubits=2, reps=1)` — a simple variational form with $R_Y$ rotations and CNOT entanglement
- **Optimizer**: `COBYLA(maxiter=500)` — a gradient-free classical optimizer 
- **Sampler**: `StatevectorSampler` for noise-free simulation

> **Note**: In the paper, "depth $d$" refers to the number of repetitions of the feature map unitary $U_{\Phi(\mathbf{x})} H^{\otimes n}$. The variational ansatz $W(\boldsymbol{\theta})$ is a separate circuit appended after the feature map.

In [ ]:
# 3.2 VQC training across all depths and dataset splits
# This cell may take several minutes to run (15 configurations × 200 optimizer iterations each)

vqc_results = {}
sampler = StatevectorSampler()

# Use a single set for the detailed depth sweep first (Set I)
set_name = "I"

for d in depths:
    data = variational_data[(set_name, d)]
    X_train, y_train_vqc = data["X_train"], data["y_train"]
    X_test, y_test_vqc = data["X_test"], data["y_test"]
    
    # VQC requires labels as 0/1 (not +1/-1)
    y_train_01 = (y_train_vqc + 1) // 2  # map {-1,+1} -> {0,1}
    y_test_01 = (y_test_vqc + 1) // 2
    
    # Feature map with depth d
    if d == 0:
        # d=0 means no data encoding. ZZFeatureMap(reps=0) has no parameters,
        # causing VQC to reject 2D input (num_inputs=0). Instead, use a trivial
        # feature map: RZ rotations on |0⟩ only add a global phase, so the state
        # remains |0⟩ — matching the paper's "no encoding" baseline.
        x = ParameterVector("x", 2)
        feature_map = QuantumCircuit(2)
        for i in range(2):
            feature_map.rz(x[i], i)
    else:
        feature_map = ZZFeatureMap(feature_dimension=2, reps=d)
    
    # Variational ansatz
    ansatz = RealAmplitudes(num_qubits=2, reps=1)
    
    # Optimizer
    optimizer = COBYLA(maxiter=500)
    
    # Callback to track loss
    loss_history = []
    def callback(weights, loss):
        loss_history.append(loss)
    
    # Build and train VQC
    vqc = VQC(
        feature_map=feature_map,
        ansatz=ansatz,
        optimizer=optimizer,
        sampler=sampler,
        callback=callback
    )
    
    vqc.fit(X_train, y_train_01)
    
    # Evaluate
    train_acc = vqc.score(X_train, y_train_01)
    test_acc = vqc.score(X_test, y_test_01)
    paper_acc = data["paper_accuracy"]
    
    vqc_results[d] = {
        "vqc": vqc,
        "train_accuracy": train_acc,
        "test_accuracy": test_acc,
        "paper_accuracy": paper_acc,
        "loss_history": loss_history.copy()
    }
    
    print(f"  d={d}: train_acc={train_acc:.4f}, test_acc={test_acc:.4f}, paper_acc={paper_acc:.4f}, "
          f"final_loss={loss_history[-1]}")

print("\nVQC training complete for Set I.")

In [ ]:
# 3.3 Plot training loss convergence for each depth
fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.viridis(np.linspace(0, 1, len(depths)))

for d, color in zip(depths, colors):
    loss_hist = vqc_results[d]["loss_history"]
    ax.plot(range(1, len(loss_hist) + 1), loss_hist, label=f"d={d}", color=color, linewidth=1.5)

ax.set_xlabel("Optimizer Iteration", fontsize=12)
ax.set_ylabel("Loss (Cross-Entropy)", fontsize=12)
ax.set_title("VQC Training Loss Convergence — Set I", fontsize=14)
ax.legend(title="Feature Map Depth", fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 3.4 Accuracy vs. Depth — Comparison with Paper

The central result of the paper's variational experiments is that classification accuracy peaks at depth $d=2$ and does not improve further. We plot our VQC accuracy alongside the paper's reported accuracy for each depth.

In [ ]:
# 3.4 Accuracy vs. depth comparison
our_test_accs = [vqc_results[d]["test_accuracy"] for d in depths]
paper_accs = [vqc_results[d]["paper_accuracy"] for d in depths]
our_train_accs = [vqc_results[d]["train_accuracy"] for d in depths]

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(depths, our_train_accs, 'o-', label="Our VQC (train)", color="#2ecc71", linewidth=2, markersize=8)
ax.plot(depths, our_test_accs, 's-', label="Our VQC (test)", color="#3498db", linewidth=2, markersize=8)
ax.plot(depths, paper_accs, 'D--', label="Paper (test)", color="#e74c3c", linewidth=2, markersize=8)
ax.set_xlabel("Feature Map Depth $d$", fontsize=12)
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title("VQC Accuracy vs. Feature Map Depth — Set I", fontsize=14)
ax.set_xticks(depths)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print table
print("\n" + "=" * 60)
print(f"{'Depth d':<10} {'Our Train':>12} {'Our Test':>12} {'Paper Test':>12}")
print("=" * 60)
for d in depths:
    print(f"{d:<10} {vqc_results[d]['train_accuracy']:>12.4f} "
          f"{vqc_results[d]['test_accuracy']:>12.4f} "
          f"{vqc_results[d]['paper_accuracy']:>12.4f}")
print("=" * 60)

💡 **Explanation:** The accuracy vs. depth plot is the key figure from the paper. Expected behavior:
- **d=0**: Low accuracy — the variational ansatz alone (without data encoding) cannot separate the classes
- **d=1**: Moderate accuracy — the first-order feature map is classically simulable and provides limited expressiveness
- **d=2**: Peak accuracy — the second-order entangling feature map provides the right level of expressiveness
- **d≥3**: No significant improvement (or slight degradation) — more depth doesn't help because the additional expressiveness is redundant for this problem, and on real hardware would add noise

The paper's test accuracy curve should closely match ours if the simulation is faithful.

### 3.5 VQC training across ALL dataset splits (Sets I, II, III)

In [ ]:
# 3.5 VQC training across ALL dataset splits (Sets I, II, III)
# This cell trains 15 VQC models — expect ~10-15 minutes runtime

all_vqc_results = {}
sampler = StatevectorSampler()

for set_name in set_names:
    for d in depths:
        data = variational_data[(set_name, d)]
        X_train, y_train_raw = data["X_train"], data["y_train"]
        X_test, y_test_raw = data["X_test"], data["y_test"]
        
        # Convert labels to 0/1 for VQC
        y_train_01 = (y_train_raw + 1) // 2
        y_test_01 = (y_test_raw + 1) // 2
        
        # Feature map with depth d
        if d == 0:
            x = ParameterVector("x", 2)
            feature_map = QuantumCircuit(2)
            for i in range(2):
                feature_map.rz(x[i], i)
        else:
            feature_map = ZZFeatureMap(feature_dimension=2, reps=d)
        ansatz = RealAmplitudes(num_qubits=2, reps=1)
        optimizer = COBYLA(maxiter=500)
        
        loss_history = []
        def callback(weights, loss):
            loss_history.append(loss)
        
        vqc = VQC(
            feature_map=feature_map,
            ansatz=ansatz,
            optimizer=optimizer,
            sampler=sampler,
            callback=callback
        )
        
        vqc.fit(X_train, y_train_01)
        
        train_acc = vqc.score(X_train, y_train_01)
        test_acc = vqc.score(X_test, y_test_01)
        
        all_vqc_results[(set_name, d)] = {
            "train_accuracy": train_acc,
            "test_accuracy": test_acc,
            "paper_accuracy": data["paper_accuracy"],
            "loss_history": loss_history.copy()
        }
        
        print(f"  Set {set_name}, d={d}: test_acc={test_acc:.4f}, paper={data['paper_accuracy']:.4f}")

print("\nAll VQC training complete.")

### 3.6 Aggregated Results — Mean Accuracy Across All Splits

Following the paper's methodology, we average the test accuracy across the three dataset splits for each depth to get a more robust estimate.

In [ ]:
# 3.6 Aggregated results — mean accuracy across all splits
mean_our_test = []
mean_paper_test = []
mean_our_train = []

for d in depths:
    our_tests = [all_vqc_results[(s, d)]["test_accuracy"] for s in set_names]
    paper_tests = [all_vqc_results[(s, d)]["paper_accuracy"] for s in set_names]
    our_trains = [all_vqc_results[(s, d)]["train_accuracy"] for s in set_names]
    mean_our_test.append(np.mean(our_tests))
    mean_paper_test.append(np.mean(paper_tests))
    mean_our_train.append(np.mean(our_trains))

# Plot
fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(depths, mean_our_train, 'o-', label="Our VQC (train, mean)", color="#2ecc71", linewidth=2, markersize=8)
ax.plot(depths, mean_our_test, 's-', label="Our VQC (test, mean)", color="#3498db", linewidth=2, markersize=8)
ax.plot(depths, mean_paper_test, 'D--', label="Paper (test, mean)", color="#e74c3c", linewidth=2, markersize=8)
ax.set_xlabel("Feature Map Depth $d$", fontsize=12)
ax.set_ylabel("Accuracy (mean over 3 splits)", fontsize=12)
ax.set_title("VQC Accuracy vs. Depth — Averaged Across All Splits", fontsize=14)
ax.set_xticks(depths)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Table
print("\n" + "=" * 70)
print(f"{'Depth d':<10} {'Our Train (mean)':>18} {'Our Test (mean)':>18} {'Paper Test (mean)':>18}")
print("=" * 70)
for i, d in enumerate(depths):
    print(f"{d:<10} {mean_our_train[i]:>18.4f} {mean_our_test[i]:>18.4f} {mean_paper_test[i]:>18.4f}")
print("=" * 70)

### 3.7 VQC Decision Boundaries at Selected Depths

We visualize the decision boundaries for the VQC at depths $d=0, 2, 4$ on Set I to see how the classifier's separation improves with depth.

In [ ]:
# 3.7 VQC decision boundaries at selected depths (d=0, 2, 4) for Set I
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
selected_depths = [0, 2, 4]

for idx, d in enumerate(selected_depths):
    vqc_model = vqc_results[d]["vqc"]
    data = variational_data[("I", d)]
    X_train = data["X_train"]
    y_train_01 = (data["y_train"] + 1) // 2
    
    x_min, x_max = X_train[:, 0].min() - 0.5, X_train[:, 0].max() + 0.5
    y_min, y_max = X_train[:, 1].min() - 0.5, X_train[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 30), np.linspace(y_min, y_max, 30))
    grid_points = np.c_[xx.ravel(), yy.ravel()]
    Z = vqc_model.predict(grid_points).reshape(xx.shape)
    
    axes[idx].contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.RdBu)
    axes[idx].scatter(X_train[:, 0], X_train[:, 1], c=y_train_01, cmap=plt.cm.RdBu, edgecolors="k", s=40)
    acc = vqc_results[d]["test_accuracy"]
    axes[idx].set_title(f"d={d} (test acc={acc:.2f})", fontsize=12)
    axes[idx].set_xlabel("Feature $x_1$")
    axes[idx].set_ylabel("Feature $x_2$")

plt.suptitle("VQC Decision Boundaries — Set I at Selected Depths", fontsize=14)
plt.tight_layout()
plt.show()

---
## Part 4: Comparison & Analysis

We now compare the two classification strategies — Quantum Kernel Estimator and Quantum Variational Classifier — and discuss the key findings from the paper.

### 4.1 Kernel Estimator vs. Variational Classifier — Accuracy Comparison

Both methods use the same feature map $\mathcal{U}_{\Phi(\mathbf{x})}$ but differ in how they leverage it. The kernel method computes the full Gram matrix and delegates classification to a classical SVM, while the variational method trains a parametrized circuit from scratch.

In [ ]:
# 4.1 Side-by-side comparison: Kernel Estimator vs Variational Classifier
# Kernel method uses ZZFeatureMap d=2; VQC uses ZZFeatureMap d=2 with RealAmplitudes ansatz

print("=" * 75)
print(f"{'Method':<30} {'Set I':>10} {'Set II':>10} {'Set III':>10} {'Mean':>10}")
print("=" * 75)

# Kernel Estimator results (from Part 2)
kernel_accs = [kernel_results[s]["our_accuracy"] for s in set_names]
print(f"{'Kernel Estimator (d=2)':<30} " + " ".join(f"{a:>10.4f}" for a in kernel_accs) + f" {np.mean(kernel_accs):>10.4f}")

# VQC results at d=2 (from Part 3)
vqc_d2_accs = [all_vqc_results[(s, 2)]["test_accuracy"] for s in set_names]
print(f"{'Variational Classifier (d=2)':<30} " + " ".join(f"{a:>10.4f}" for a in vqc_d2_accs) + f" {np.mean(vqc_d2_accs):>10.4f}")

# Paper's kernel results
paper_kernel = [kernel_results[s]["paper_accuracy"] for s in set_names]
print(f"{'Paper Kernel (d=2)':<30} " + " ".join(f"{a:>10.4f}" for a in paper_kernel) + f" {np.mean(paper_kernel):>10.4f}")

# Paper's variational results at d=2
paper_vqc_d2 = [all_vqc_results[(s, 2)]["paper_accuracy"] for s in set_names]
print(f"{'Paper Variational (d=2)':<30} " + " ".join(f"{a:>10.4f}" for a in paper_vqc_d2) + f" {np.mean(paper_vqc_d2):>10.4f}")

print("=" * 75)

# Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 6))
methods = ["Kernel\nEstimator", "Variational\nClassifier (d=2)"]
our_means = [np.mean(kernel_accs), np.mean(vqc_d2_accs)]
paper_means = [np.mean(paper_kernel), np.mean(paper_vqc_d2)]
x = np.arange(len(methods))
width = 0.3

bars1 = ax.bar(x - width/2, our_means, width, label="Our Implementation", color="#3498db")
bars2 = ax.bar(x + width/2, paper_means, width, label="Paper Results", color="#e74c3c", alpha=0.7)

ax.set_ylabel("Mean Test Accuracy", fontsize=12)
ax.set_title("Kernel Estimator vs. Variational Classifier — Mean Accuracy", fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(methods, fontsize=11)
ax.set_ylim(0, 1.1)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f"{bar.get_height():.2f}", ha='center', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f"{bar.get_height():.2f}", ha='center', fontsize=10)

plt.tight_layout()
plt.show()

### 4.2 Why Depth $d=2$ Is the Sweet Spot

The paper identifies depth $d=2$ as the optimal feature map depth. This is a central claim with both theoretical and practical justification:

1. **$d=1$ is classically simulable**: At depth 1, the feature map circuit produces product states (for first-order) or shallow entangled states that can be efficiently sampled classically via the technique of uniform sampling (Havlicek *et al.*, Supplementary Section IV). No quantum advantage is possible.

2. **$d=2$ provides genuine quantum correlations**: The second repetition of $U_{\Phi(\mathbf{x})} H^{\otimes n}$ creates entanglement patterns that are conjectured to be hard to simulate classically. The IQP-like structure (Instantaneous Quantum Polynomial) at $d=2$ falls under the conjecture of quantum computational supremacy.

3. **$d \geq 3$ adds no benefit**: While deeper circuits are more expressive, the additional expressiveness is redundant for this classification task. On real quantum hardware, deeper circuits accumulate more noise (gate errors, decoherence), which degrades performance. The paper's experimental results on ibmq devices confirm this.

In [ ]:
# 4.2 Visualize the depth d=2 "sweet spot" effect
fig, ax = plt.subplots(figsize=(9, 6))

# VQC mean test accuracy at each depth
ax.plot(depths, mean_our_test, 's-', label="VQC test (mean)", color="#3498db", linewidth=2.5, markersize=10)
ax.axvline(x=2, color='#e74c3c', linestyle='--', linewidth=2, label='Optimal depth d=2')
ax.fill_between([1.5, 2.5], 0, 1.05, alpha=0.1, color='#2ecc71', label='Classically hard regime')

# Annotate regions
ax.annotate('Classically\nsimulable', xy=(0.5, 0.15), fontsize=10, ha='center', color='gray',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))
ax.annotate('Potential\nquantum advantage', xy=(2, 0.15), fontsize=10, ha='center', color='#2ecc71',
            fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgreen', alpha=0.5))
ax.annotate('Diminishing returns\n(+ noise on hardware)', xy=(3.5, 0.15), fontsize=10, ha='center', color='gray',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

ax.set_xlabel("Feature Map Depth $d$", fontsize=12)
ax.set_ylabel("Mean Test Accuracy", fontsize=12)
ax.set_title("The Depth $d=2$ Sweet Spot", fontsize=14)
ax.set_xticks(depths)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 4.3 Effect of Data Map Functions on Classification

The choice of $\phi_S(\mathbf{x})$ — how classical data is encoded into rotation angles — significantly impacts classification performance. We compare the default, product, and sin data maps from Part 1 on the paper's dataset.

In [ ]:
# 4.3 Compare data map functions on paper dataset (Set I)
data_map_variants = {
    "Default (pi-x_i)(pi-x_j)": None,  # None = use ZZFeatureMap default
    "Product (pi-x_i)": product_data_map_func,
    "Sin sin(pi-x_i)": sin_data_map_func,
}

print("Data Map Function Comparison on Paper Set I:")
print("=" * 55)

data_map_accs = {}
for name, data_map in data_map_variants.items():
    if data_map is None:
        fm = ZZFeatureMap(feature_dimension=2, reps=2, entanglement='full')
    else:
        fm = ZZFeatureMap(feature_dimension=2, reps=2, entanglement='full', data_map_func=data_map)
    
    sampler = StatevectorSampler()
    fidelity = ComputeUncompute(sampler=sampler)
    qk = FidelityQuantumKernel(feature_map=fm, fidelity=fidelity)
    qsvc = QSVC(quantum_kernel=qk)
    qsvc.fit(kernel_sets["I"]["X_train"], kernel_sets["I"]["y_train"])
    acc = qsvc.score(kernel_sets["I"]["X_test"], kernel_sets["I"]["y_test"])
    data_map_accs[name] = acc
    print(f"  {name:<30} Accuracy: {acc:.4f}")

print("=" * 55)

# Bar chart
fig, ax = plt.subplots(figsize=(8, 5))
names = list(data_map_accs.keys())
accs = list(data_map_accs.values())
bars = ax.barh(names, accs, color=['#2ecc71', '#3498db', '#9b59b6'])
ax.set_xlim(0, 1.05)
ax.set_xlabel("Test Accuracy")
ax.set_title("Effect of Data Map Function — Paper Set I (ZZFeatureMap d=2)")
for bar, acc in zip(bars, accs):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, f"{acc:.2f}", va='center')
plt.tight_layout()
plt.show()

### 4.4 Classical Simulability & Quantum Advantage

The potential quantum advantage of these methods rests on the conjecture that certain quantum feature maps are hard to simulate classically. Specifically:

- **First-order maps** 

- **Second-order maps** 

- **The kernel method** 

- **The variational method**

### References